# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 |
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [1]:
!pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")

Setup complete!


In [2]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()


# from google.colab import userdata

# First, set it into the environment
# os.environ["OPENROUTER_API_KEY"] = userdata.get('MY_OPEN_ROUTER_KEY')
os.environ["OPENROUTER_API_KEY"] = "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"

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY","")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [3]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence
# TODO: Predict on llm reviews, keep where confidence >= 0.65 AND prediction matches LLM label
x_llm=vec.transform(llm["review"])
pred=clf.predict(x_llm)
probs=clf.predict_proba(x_llm)
confidence=probs.max(axis=1)
llm["pred"]=pred
llm["confidence"]=confidence
filtered_llm=llm[(llm["pred"]==llm["label"])&(llm["confidence"]>=0.65)]
print("Filtered LLM samples:", len(filtered_llm))

#  1d. Merge & deduplicate
# TODO: Combine gold + weak + filtered_llm, drop_duplicates on "review"
# Save as consolidated_base.csv
consolidated_base=pd.concat([gold,weak,filtered_llm[["review","label"]]])
consolidated_base=consolidated_base.drop_duplicates(subset="review")
consolidated_base.to_csv("consolidated_base.csv",index=False)
print(len(consolidated_base))
#  1e. Class distribution analysis
# TODO: Count per class, identify minority, plot and save class_distribution.png
counts = consolidated_base["label"].value_counts()

# Identify minority class
minority_class = counts.idxmin()

# Plot distribution
counts.plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.savefig("class_distribution.png")

print("Class Counts:\n", counts)
print("Minority class:", minority_class)
print("Minority class count:", counts.min())

Gold: 100, Weak: 220, LLM: 150
Filtered LLM samples: 27
328
Class Counts:
 label
Negative    151
Neutral     115
Positive     62
Name: count, dtype: int64
Minority class: Positive
Minority class count: 62


In [4]:
#  1f. Augmentation functions
from nltk.wsd import lesk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    # TODO: Implement using nltk.corpus.wordnet, nltk.pos_tag, word_tokenize
    tokens=word_tokenize(text) #divides all words into list,from nltk
    tagged=pos_tag(tokens) # assigns tag to each word,from nltk
    n_replace=max(1,int(len(tokens)*replace_frac))
    candidate_ids=list(range(len(tokens)))
    random.shuffle(candidate_ids)
    new_tokens=tokens.copy()
    replaced=0
    for i in candidate_ids:
      word=tokens[i].lower()
      if word in PRESERVE_WORDS:
        continue
      pos=None
      if tagged[i][1].startswith("J"):
        pos=wordnet.ADJ
      elif tagged[i][1].startswith("V"):
        pos=wordnet.VERB
      elif tagged[i][1].startswith("N"):
        pos=wordnet.NOUN
      elif tagged[i][1].startswith("R"):
        pos=wordnet.ADV
      if pos is None:
        continue
      synsets=wordnet.synsets(word,pos=pos)
      if not synsets:
        continue
      # Use ONLY first synset (most relevant)
      synonyms = set()
      for lemma in synsets[0].lemmas():
          synonym = lemma.name().replace("_", " ")

            # Avoid same word replacement
          if synonym.lower() != word:
              synonyms.add(synonym)
      if synonyms:
        new_tokens[i]=random.choice(list(synonyms))
        replaced+=1
      if replaced >= n_replace:
            break

    return " ".join(new_tokens)


def back_translate(text, src="en", mid="hi"):
    """Translate English -> Hindi -> English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    # TODO: Implement with error handling and rate-limit sleep
    try:
      #English to Hindi
      translated=GoogleTranslator(source=src,target=mid).translate(text)
      #small delay to avoid hitting ratelimits
      time.sleep(1)
      #back translate, Hindi to English
      back_translated=GoogleTranslator(source=mid,target=src).translate(translated)
      return back_translated
    except Exception as e:
      print("Back translation is failed",e)
      return text


def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30–0.95)."""
    # TODO: Implement Jaccard similarity check
    if augmented is None:return False
    tokens1=set(word_tokenize(original.lower()))
    tokens2=set(word_tokenize(augmented.lower()))
    intersection=len(tokens1.intersection(tokens2))
    union=len(tokens1.union(tokens2))
    if union==0: return False
    jaccard_score=intersection/union

    return 0.3<=jaccard_score<=0.95



#  1g. Apply augmentation to minority class
df=pd.read_csv("consolidated_base.csv")
minority_df=df[df["label"]==minority_class]
# TODO: For each minority-class sample, generate 2 augmented versions (one per method)
augmented_rows=[]
for _,row in minority_df.iterrows():
  text=row["review"]
  label=row["label"]
  #synonym replacement-method 1
  syn_aug=synonym_replacement(text)
  if quality_filter(text,syn_aug):
    augmented_rows.append({"review":syn_aug,"label":label})
  #back translation-method 2
  back_aug=back_translate(text)
  if quality_filter(text,back_aug):
    augmented_rows.append({"review":back_aug,"label":label})

# TODO: Apply quality_filter, keep only passing samples
augmented_df=pd.DataFrame(augmented_rows)
print("Augmented samples:", len(augmented_df))

# TODO: Save augmented_classical.csv
augmented_df.to_csv("augmented_classical.csv", index=False)


Augmented samples: 123


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [5]:
from openai import OpenAI

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

#  2a. Design your few-shot prompt
# TODO: Build a prompt string with 3-4 example reviews from gold standard
examples=gold.sample(4,random_state=20)
example_text=""
for _,row in examples.iterrows():
  example_text+=f'Review:{row["review"]} | Sentiment:{row["label"]}\n'
print(example_text)
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]

PROMPT_TEMPLATE = f"""You are a movie review generator. ...

Generate Realistic movie reviews with sentiment labels.

Example reviews:
{example_text}

Instructions:
Generate movie reviews and return only JSON in this format:
[
  {{"review": "...", "sentiment": "Positive", "movie": "..."}},
  {{"review": "...", "sentiment": "Negative", "movie": "..."}}
]
Sentiment must be one of: Positive, Negative, Neutral.
"""

# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

#  2b. Generate reviews in batches
generated_reviews=[]
target=300
batch=20
TARGET_COUNTS={
    "Positive":150,
    "Negative":100,
    "Neutral":50
}
sentiment_counts={"Positive":0,"Negative":0,"Neutral":0}
# TODO: Loop to generate ~300 reviews in batches of 20
import json
while (len(generated_reviews)<target):
  try:
    response=client.chat.completions.create(model=LLM_MODEL,messages=[{"role":"user","content":PROMPT_TEMPLATE+f"\nGenerate {batch} reviews"}],temperature=0.9)
    content=response.choices[0].message.content
    json_match = re.search(r"\[.*\]", content, re.DOTALL)

    if json_match:
      data = json.loads(json_match.group())

    else:
      raise ValueError("No JSON found in response")
    for item in data:
      #print(item)
      if(len(generated_reviews)>=target):break
      # Only accept if we need this sentiment
      sentiment=item.get("sentiment")
      if sentiment in sentiment_counts.keys() and sentiment_counts[sentiment] < TARGET_COUNTS[sentiment]:

        generated_reviews.append(item)
        sentiment_counts[sentiment]+=1
    print("Generated:",len(generated_reviews))
  except Exception as e:
    print("Error:",e)
  time.sleep(0.5)
generated_reviews=generated_reviews[:300]
llm_df=pd.DataFrame(generated_reviews)
#c=llm_df["sentiment"].value_counts()
#print(c)


# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors


#  2c. Diversity metrics
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score
from nltk.translate.bleu_score import sentence_bleu,SmoothingFunction

smooth=SmoothingFunction().method1

def compute_self_bleu(texts):

    scores=[]

    for i in range(len(texts)):

        refs=[texts[j].split() for j in range(len(texts)) if j!=i]
        #print ('refs:',refs)

        hyp=texts[i].split()
        #print('hyp:',hyp)

        if len(refs)>0:
            score=sentence_bleu(refs,hyp,smoothing_function=smooth)
            scores.append(score)

    if len(scores)==0:
        return 0

    return np.mean(scores)


diversity_results={}

for sentiment in ["Positive","Negative","Neutral"]:

    subset=llm_df[llm_df["sentiment"]==sentiment]["review"].tolist()
    #print(subset)

    if len(subset)>1:
        diversity_results[sentiment]=compute_self_bleu(subset)
    else:
        diversity_results[sentiment]=None

print(diversity_results)

#  2d. Sentiment consistency check
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv
X_gen=vec.transform(llm_df["review"])

preds=clf.predict(X_gen)

llm_df["predicted_sentiment"]=preds

flagged=llm_df[llm_df["predicted_sentiment"]!=llm_df["sentiment"]]

print("Flagged samples:",len(flagged))

flagged.to_csv("llm_generated_flagged.csv",index=False)

#  2e. Save outputs
# TODO: Save llm_generated_300.csv and diversity_report.txt
llm_df.to_csv("llm_generated_300.csv",index=False)

with open("diversity_report.txt","w") as f:

    for k,v in diversity_results.items():
        f.write(f"{k} Self-BLEU: {v}\n")

print("Files saved.")

Review:Great concept, but the execution was clichéd. I'm honestly still trying to process what I just saw. | Sentiment:Neutral
Review:I'm honestly still trying to process what I just saw. I respect the ambition, even if it didn't fully land. | Sentiment:Neutral
Review:I was completely blown away by this film. The lead actor delivered a refreshing performance. I will be thinking about this for days. | Sentiment:Positive
Review:I am confused about how I feel. A polarizing experience for sure. | Sentiment:Neutral

Generated: 20
Generated: 40
Generated: 60
Generated: 80
Generated: 100
Generated: 120
Generated: 140
Generated: 160
Generated: 180
Generated: 200
Generated: 220
Generated: 236
Generated: 251
Generated: 264
Generated: 271
Generated: 278
Generated: 285
Generated: 294
Generated: 300
{'Positive': 0.4995657512205844, 'Negative': 0.4429638104305389, 'Neutral': 0.27635822810662913}
Flagged samples: 147
Files saved.


## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [6]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

#  3a. Strategic sampling
# TODO: From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)
cons_df = pd.read_csv("consolidated_base.csv")
cons_df["text_len"] = cons_df["review"].str.len()
cons_df = cons_df.sort_values("text_len")

sample_pos = cons_df[cons_df["label"]=="Positive"].head(40)
sample_neg = cons_df[cons_df["label"]=="Negative"].head(40)
sample_neu = cons_df[cons_df["label"]=="Neutral"].head(20)

sample_100 = pd.concat([sample_pos, sample_neg, sample_neu]).reset_index(drop=True)
sample_100 = sample_100.drop(columns=["text_len"])
print(f"Sampled: {len(sample_100)} reviews")
print(sample_100["label"].value_counts())

#  3b. Translation pipeline
# TODO: Translate each review English -> Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits
hindi_translations = []
for idx, row in sample_100.iterrows():
    try:
        translated = GoogleTranslator(source='en', target='hi').translate(row['review'])
        hindi_translations.append(translated)
    except Exception as e:
        print(f"Translation failed for index {idx}: {e}")
        hindi_translations.append(None)
    time.sleep(0.5)

sample_100["hindi"] = hindi_translations
print(f"Translated {sum(1 for h in hindi_translations if h is not None)} reviews to Hindi")

#  3c. Back-translation & BLEU
# TODO: Translate Hindi -> English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"
smooth = SmoothingFunction().method1
back_translations = []
bleu_scores = []
quality_flags = []

for idx, row in sample_100.iterrows():
    if row["hindi"] is None:
        back_translations.append(None)
        bleu_scores.append(0.0)
        quality_flags.append("FAIL")
        continue
    try:
        back_trans = GoogleTranslator(source='hi', target='en').translate(row['hindi'])
        back_translations.append(back_trans)
    except Exception as e:
        print(f"Back-translation failed for index {idx}: {e}")
        back_translations.append(None)
        bleu_scores.append(0.0)
        quality_flags.append("FAIL")
        time.sleep(0.5)
        continue

    # Compute BLEU score
    ref_tokens = word_tokenize(row["review"].lower())
    hyp_tokens = word_tokenize(back_trans.lower())
    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smooth)
    bleu_scores.append(bleu)
    quality_flags.append("PASS" if bleu >= 0.3 else "FAIL")
    time.sleep(0.5)

sample_100["back_translated"] = back_translations
sample_100["bleu_score"] = bleu_scores
sample_100["quality_flag"] = quality_flags

print(f"PASS: {quality_flags.count('PASS')}, FAIL: {quality_flags.count('FAIL')}")
print(f"Mean BLEU: {np.mean(bleu_scores):.4f}")

#  3d. Sentiment preservation check
# TODO: Predict sentiment on back-translated text, compare with original label
valid_back = sample_100[sample_100["back_translated"].notna()].copy()
X_back = vec.transform(valid_back["back_translated"])
back_preds = clf.predict(X_back)
valid_back["back_pred"] = back_preds
sentiment_match = (valid_back["back_pred"] == valid_back["label"]).sum()
print(f"Sentiment preserved: {sentiment_match}/{len(valid_back)} ({sentiment_match/len(valid_back):.2%})")

#  3e. Manual verification
# TODO: Print 5 random samples for inspection
manual_samples = sample_100.sample(5, random_state=RANDOM_SEED)
for _, row in manual_samples.iterrows():
    print("="*60)
    print(f"Original ({row['label']}): {row['review']}")
    print(f"Hindi: {row['hindi']}")
    print(f"Back-translated: {row['back_translated']}")
    print(f"BLEU: {row['bleu_score']:.4f} | Quality: {row['quality_flag']}")

#  3f. Save
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag
bilingual_df = sample_100[["review", "label", "hindi", "back_translated", "bleu_score", "quality_flag"]]
bilingual_df.to_csv("bilingual_reviews.csv", index=False)
print(f"\nSaved bilingual_reviews.csv with {len(bilingual_df)} rows")

Sampled: 100 reviews
label
Positive    40
Negative    40
Neutral     20
Name: count, dtype: int64
Translated 100 reviews to Hindi
PASS: 91, FAIL: 9
Mean BLEU: 0.6303
Sentiment preserved: 87/100 (87.00%)
Original (Neutral): It doesn't leave a lasting impression. It was... fine.
Hindi: यह कोई स्थायी प्रभाव नहीं छोड़ता. यह... ठीक था.
Back-translated: It does not leave any lasting effect. It was...okay.
BLEU: 0.2545 | Quality: FAIL
Original (Negative): Total garbage. Save your money and watch something else.
Hindi: कुल कचरा. अपना पैसा बचाएं और कुछ और देखें।
Back-translated: Total garbage. Save your money and watch something else.
BLEU: 1.0000 | Quality: PASS
Original (Negative): A refreshing take on a tired genre. I can't wait to see it again.
Hindi: एक थकी हुई शैली पर एक ताज़ा प्रस्तुति। मैं इसे दोबारा देखने के लिए इंतजार नहीं कर सकता.
Back-translated: A refreshing take on a tired genre. I can't wait to see it again.
BLEU: 1.0000 | Quality: PASS
Original (Negative): I want my two hours ba

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [11]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class)
# TODO: Sample from consolidated_base, mix short and long reviews
audio_base = pd.read_csv("consolidated_base.csv")
audio_base["review"] = audio_base["review"].astype(str)
audio_base["text_len"] = audio_base["review"].str.len()

selected_parts = []
for cls in ["Positive", "Negative", "Neutral"]:
    cls_df = audio_base[audio_base["label"] == cls].sort_values("text_len").copy()
    target_n = min(10, len(cls_df))
    if target_n == 0:
        continue

    if target_n <= 2:
        class_pick = cls_df.head(target_n)
    else:
        short_n = target_n // 2
        long_n = target_n - short_n
        class_pick = pd.concat([cls_df.head(short_n), cls_df.tail(long_n)], ignore_index=True)
        class_pick = class_pick.drop_duplicates(subset="review")

        if len(class_pick) < target_n:
            needed = target_n - len(class_pick)
            extra = cls_df[~cls_df["review"].isin(class_pick["review"])].head(needed)
            class_pick = pd.concat([class_pick, extra], ignore_index=True)

    selected_parts.append(class_pick.head(target_n))

audio_sample_df = pd.concat(selected_parts, ignore_index=True)
audio_sample_df = audio_sample_df[["review", "label"]].drop_duplicates(subset="review").reset_index(drop=True)
audio_sample_df = audio_sample_df.head(30).copy()
audio_sample_df["sample_id"] = [f"sample_{i:03d}" for i in range(1, len(audio_sample_df) + 1)]
print("Selected samples:", len(audio_sample_df))
print(audio_sample_df["label"].value_counts())

#  4b. TTS generation
os.makedirs("audio_samples", exist_ok=True)

# TODO: For each review, generate audio with gTTS (tld="com")
# Save as mp3, then load with librosa and re-save as .wav via soundfile
audio_records = []
for _, row in audio_sample_df.iterrows():
    sample_id = row["sample_id"]
    review_text = row["review"]
    label = row["label"]
    mp3_path = os.path.join("audio_samples", f"{sample_id}.mp3")
    wav_path = os.path.join("audio_samples", f"{sample_id}.wav")

    try:
        tts = gTTS(text=review_text, lang="en", tld="com")
        tts.save(mp3_path)

        y, sr = librosa.load(mp3_path, sr=16000)
        sf.write(wav_path, y, sr)

        audio_records.append({
            "sample_id": sample_id,
            "review": review_text,
            "label": label,
            "mp3_path": mp3_path,
            "wav_path": wav_path,
            "status": "ok",
            "error": "",
        })
    except Exception as e:
        print(f"Audio generation failed for {sample_id}: {e}")
        audio_records.append({
            "sample_id": sample_id,
            "review": review_text,
            "label": label,
            "mp3_path": mp3_path,
            "wav_path": wav_path,
            "status": "failed",
            "error": str(e),
        })
    time.sleep(0.2)

audio_meta_df = pd.DataFrame(audio_records)
print("Audio files generated:", (audio_meta_df["status"] == "ok").sum())

#  4c. Audio feature extraction
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv
feature_rows = []
for _, row in audio_meta_df.iterrows():
    if row["status"] != "ok" or not os.path.exists(row["wav_path"]):
        continue
    try:
        y, sr = librosa.load(row["wav_path"], sr=16000)
        duration = float(librosa.get_duration(y=y, sr=sr))
        spectral_centroid = float(librosa.feature.spectral_centroid(y=y, sr=sr).mean())
        zcr = float(librosa.feature.zero_crossing_rate(y).mean())
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        feat = {
            "sample_id": row["sample_id"],
            "label": row["label"],
            "duration": duration,
            "spectral_centroid": spectral_centroid,
            "zero_crossing_rate": zcr,
        }
        for i in range(13):
            feat[f"mfcc_{i+1}"] = float(mfcc[i].mean())
        feature_rows.append(feat)
    except Exception as e:
        print(f"Feature extraction failed for {row['sample_id']}: {e}")

audio_features_df = pd.DataFrame(feature_rows)
audio_features_df.to_csv("audio_features.csv", index=False)
print("Saved audio_features.csv with", len(audio_features_df), "rows")


 #  4d. Whisper round-trip validation
import whisper

_whisper_model = whisper.load_model("tiny")

# TODO: Transcribe each wav with Whisper
# Compute WER (word-level Levenshtein distance / reference word count)
# Flag samples with WER > 0.25
# Save audio_validation.csv
def _levenshtein_distance_words(ref_words, hyp_words):
    rows = len(ref_words) + 1
    cols = len(hyp_words) + 1
    dp = [[0] * cols for _ in range(rows)]

    for i in range(rows):
        dp[i][0] = i
    for j in range(cols):
        dp[0][j] = j

    for i in range(1, rows):
        for j in range(1, cols):
            cost = 0 if ref_words[i - 1] == hyp_words[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )
    return dp[-1][-1]


def compute_wer(reference_text, hypothesis_text):
    ref_words = word_tokenize(str(reference_text).lower())
    hyp_words = word_tokenize(str(hypothesis_text).lower())
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    edits = _levenshtein_distance_words(ref_words, hyp_words)
    return edits / len(ref_words)


validation_rows = []
for _, row in audio_meta_df.iterrows():
    if row["status"] != "ok" or not os.path.exists(row["wav_path"]):
        validation_rows.append({
            "sample_id": row["sample_id"],
            "review": row["review"],
            "label": row["label"],
            "transcription": "",
            "wer": 1.0,
            "high_wer_flag": True,
            "status": row["status"],
            "error": row["error"],
        })
        continue

    transcription = ""
    error_text = ""
    try:
        result = _whisper_model.transcribe(row["wav_path"], fp16=False)
        transcription = result.get("text", "").strip()
    except Exception as e:
        error_text = str(e)
        print(f"Whisper transcription failed for {row['sample_id']}: {e}")

    wer = compute_wer(row["review"], transcription) if transcription else 1.0
    validation_rows.append({
        "sample_id": row["sample_id"],
        "review": row["review"],
        "label": row["label"],
        "transcription": transcription,
        "wer": wer,
        "high_wer_flag": wer > 0.25,
        "status": "ok" if transcription else "failed",
        "error": error_text,
    })

audio_validation_df = pd.DataFrame(validation_rows)
audio_validation_df.to_csv("audio_validation.csv", index=False)
# Save audio_validation.csv
print("Saved audio_validation.csv with", len(audio_validation_df), "rows")
print("High WER samples:", int(audio_validation_df["high_wer_flag"].sum()))

Selected samples: 30
label
Positive    10
Negative    10
Neutral     10
Name: count, dtype: int64
Audio files generated: 30
Saved audio_features.csv with 30 rows
Saved audio_validation.csv with 30 rows
High WER samples: 3


## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [12]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset
# TODO: Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv
consolidated_base = pd.read_csv("consolidated_base.csv")
augmented_classical = pd.read_csv("augmented_classical.csv")
llm_generated = pd.read_csv("llm_generated_300.csv")
llm_flagged = pd.read_csv("llm_generated_flagged.csv")
bilingual_reviews = pd.read_csv("bilingual_reviews.csv")

if "sentiment" in llm_generated.columns and "label" not in llm_generated.columns:
    llm_generated = llm_generated.rename(columns={"sentiment": "label"})
llm_generated = llm_generated[["review", "label"]].copy()

flagged_reviews = set(llm_flagged["review"].astype(str)) if "review" in llm_flagged.columns else set()
llm_clean = llm_generated[~llm_generated["review"].astype(str).isin(flagged_reviews)].copy()

bilingual_english = bilingual_reviews[["review", "label"]].copy()

final_augmented = pd.concat(
    [consolidated_base[["review", "label"]], augmented_classical[["review", "label"]], llm_clean, bilingual_english],
    ignore_index=True,
    sort=False,
)
final_augmented = final_augmented.dropna(subset=["review", "label"]).copy()
final_augmented["review"] = final_augmented["review"].astype(str).str.strip()
final_augmented = final_augmented[final_augmented["review"] != ""]
final_augmented = final_augmented.drop_duplicates(subset="review").reset_index(drop=True)
final_augmented.to_csv("final_augmented_dataset.csv", index=False)
print("Saved final_augmented_dataset.csv with", len(final_augmented), "rows")


 #  5b. Black-Box Evaluation
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation (small dataset only)
# TODO: baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)
baseline_acc = evaluator.run_evaluation(
    consolidated_base,
    test_df,
    model_name="Baseline (Consolidated)",
)

# Augmented evaluation (full dataset)
# TODO: augmented_acc = evaluator.run_evaluation(final_augmented, test_df)
augmented_acc = evaluator.run_evaluation(
    final_augmented,
    test_df,
    model_name="Augmented (Final Dataset)",
)

# Print comparison
print(f"Baseline accuracy:  {baseline_acc:.2%}")
print(f"Augmented accuracy: {augmented_acc:.2%}")
print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")

Saved final_augmented_dataset.csv with 602 rows
Initializing Black-Box Embedder...
Embedder loaded successfully.

--- Evaluating: Baseline (Consolidated) ---
Training on 228 samples (excluded 100 test overlaps)...
Accuracy: 76.00%
Classification Report:
              precision    recall  f1-score   support

    Negative       0.53      0.84      0.65        25
     Neutral       0.86      0.68      0.76        47
    Positive       1.00      0.82      0.90        28

    accuracy                           0.76       100
   macro avg       0.80      0.78      0.77       100
weighted avg       0.82      0.76      0.77       100

----------------------------------------

--- Evaluating: Augmented (Final Dataset) ---
Training on 502 samples (excluded 100 test overlaps)...
Accuracy: 80.00%
Classification Report:
              precision    recall  f1-score   support

    Negative       0.59      0.92      0.72        25
     Neutral       0.94      0.70      0.80        47
    Positive      